## Implementasi Algoritma C4.5 (Decision Tree) untuk Klasifikasi Level Obesitas


**Author:** Muhammad Prayoga Putra Mahardhika  


---

**Algoritma:** C4.5 (Decision Tree)  
**Dataset:** Obesity Levels Dataset (ObesityDataSet_raw_and_data_sinthetic.csv)  
**Target:** Klasifikasi tingkat obesitas (`NObeyesdad`) ke dalam 7 kelas  


---
## Import Library

Mengimpor semua library yang dibutuhkan untuk analisis data, pemodelan, dan visualisasi.

In [ ]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import warnings
warnings.filterwarnings('ignore')

# Pengaturan tampilan
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('husl')

print('✅ Semua library berhasil diimpor!')

---
# Data Understanding

Pada tahap ini, kita akan memuat dataset dan memahami struktur serta karakteristik data yang digunakan.

### 1.1 Memuat Dataset

In [2]:
# Memuat dataset
# Jika menggunakan Google Colab, upload file terlebih dahulu atau gunakan path yang sesuai
df = pd.read_csv('ObesityDataSet_raw_and_data_sinthetic.csv')

print('✅ Dataset berhasil dimuat!')
print(f'Jumlah baris: {df.shape[0]}')
print(f'Jumlah kolom: {df.shape[1]}')

# Mengubah label target (kelas obesitas) ke Bahasa Indonesia
target_map = {
    'Insufficient_Weight': 'Berat Badan Kurang',
    'Normal_Weight': 'Berat Badan Normal',
    'Overweight_Level_I': 'Kelebihan Berat I',
    'Overweight_Level_II': 'Kelebihan Berat II',
    'Obesity_Type_I': 'Obesitas Tipe I',
    'Obesity_Type_II': 'Obesitas Tipe II',
    'Obesity_Type_III': 'Obesitas Tipe III'
}
df['NObeyesdad'] = df['NObeyesdad'].map(target_map)


### 1.2 Menampilkan 5 Data Pertama

In [3]:
# Menampilkan 5 data pertama
df.head()

### 1.3 Menampilkan 5 Data Terakhir

In [4]:
# Menampilkan 5 data terakhir
df.tail()

### 1.4 Informasi Dataset (Tipe Data & Non-Null Count)

In [5]:
# Menampilkan informasi dataset
df.info()

### 1.5 Statistik Deskriptif (Data Numerik)

In [6]:
# Statistik deskriptif untuk kolom numerik
df.describe()

### 1.6 Statistik Deskriptif (Data Kategorikal)

In [7]:
# Statistik deskriptif untuk kolom kategorikal
df.describe(include='object')

### 1.7 Penjelasan Atribut Dataset

Dataset **Obesity Levels** berisi data estimasi tingkat obesitas berdasarkan kebiasaan makan dan kondisi fisik seseorang. Berikut penjelasan setiap atribut:

| No | Atribut | Tipe | Penjelasan |
|:--:|---------|------|------------|
| 1 | `Gender` | Kategorikal | Jenis kelamin (Male/Female) |
| 2 | `Age` | Numerik | Usia (dalam tahun) |
| 3 | `Height` | Numerik | Tinggi badan (dalam meter) |
| 4 | `Weight` | Numerik | Berat badan (dalam kg) |
| 5 | `family_history_with_overweight` | Kategorikal | Riwayat keluarga dengan kelebihan berat badan (yes/no) |
| 6 | `FAVC` | Kategorikal | Konsumsi makanan berkalori tinggi secara sering (yes/no) |
| 7 | `FCVC` | Numerik | Frekuensi konsumsi sayuran (skala 1-3) |
| 8 | `NCP` | Numerik | Jumlah makan utama per hari (1-4) |
| 9 | `CAEC` | Kategorikal | Konsumsi makanan di antara waktu makan (no/Sometimes/Frequently/Always) |
| 10 | `SMOKE` | Kategorikal | Kebiasaan merokok (yes/no) |
| 11 | `CH2O` | Numerik | Konsumsi air per hari (dalam liter, skala 1-3) |
| 12 | `SCC` | Kategorikal | Monitoring konsumsi kalori (yes/no) |
| 13 | `FAF` | Numerik | Frekuensi aktivitas fisik per minggu (skala 0-3) |
| 14 | `TUE` | Numerik | Waktu penggunaan perangkat teknologi per hari (dalam jam, skala 0-2) |
| 15 | `CALC` | Kategorikal | Frekuensi konsumsi alkohol (no/Sometimes/Frequently/Always) |
| 16 | `MTRANS` | Kategorikal | Moda transportasi yang digunakan (Automobile/Motorbike/Bike/Public_Transportation/Walking) |
| 17 | `NObeyesdad` | Kategorikal (Target) | Tingkat obesitas → variabel yang akan diprediksi |

**Kelas Target (`NObeyesdad`) terdiri dari 7 kategori:**
1. `Insufficient_Weight` – Berat Badan Kurang
2. `Normal_Weight` – Berat Badan Normal
3. `Overweight_Level_I` – Kelebihan Berat Badan Tingkat I
4. `Overweight_Level_II` – Kelebihan Berat Badan Tingkat II
5. `Obesity_Type_I` – Obesitas Tipe I
6. `Obesity_Type_II` – Obesitas Tipe II
7. `Obesity_Type_III` – Obesitas Tipe III


### 1.8 Distribusi Variabel Target (NObeyesdad)

In [8]:
# Menampilkan distribusi variabel target
print('Distribusi kelas target (NObeyesdad):')
print('=' * 45)
target_counts = df['NObeyesdad'].value_counts()
for label, count in target_counts.items():
    pct = count / len(df) * 100
    print(f'{label:25s} : {count:4d} ({pct:.1f}%)')
print('=' * 45)
print(f'{"TOTAL":25s} : {len(df):4d}')

In [9]:
# Visualisasi distribusi target
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar Chart
colors = sns.color_palette('husl', n_colors=len(target_counts))
bars = axes[0].barh(target_counts.index, target_counts.values, color=colors)
axes[0].set_xlabel('Jumlah Data')
axes[0].set_title('Distribusi Kelas Target (NObeyesdad)', fontsize=14, fontweight='bold')
for bar, count in zip(bars, target_counts.values):
    axes[0].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                 f'{count}', va='center', fontweight='bold')

# Pie Chart
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Proporsi Kelas Target', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 1.9 Unique Values untuk Setiap Fitur Kategorikal

In [10]:
# Menampilkan unique values untuk fitur kategorikal
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print('Fitur Kategorikal dan Nilai Uniknya:')
print('=' * 60)
for col in categorical_cols:
    unique_vals = df[col].unique()
    print(f'\n📌 {col} ({len(unique_vals)} nilai unik):')
    for val in unique_vals:
        print(f'   - {val}')

---
# Data Preprocessing

Pada tahap ini, kita akan menyiapkan data agar siap untuk diproses oleh algoritma C4.5 (Decision Tree). Proses yang dilakukan meliputi:
1. Pengecekan dan penanganan missing values
2. Pengecekan data duplikat
3. Encoding data kategorikal
4. Pembagian data training dan testing

### 2.1 Pengecekan Missing Values

In [11]:
# Mengecek missing values
print('Jumlah Missing Values per Kolom:')
print('=' * 40)
missing = df.isnull().sum()
for col, count in missing.items():
    status = '✅ Tidak ada' if count == 0 else f'⚠️ {count} missing'
    print(f'{col:35s} : {status}')

total_missing = missing.sum()
print(f'\nTotal missing values: {total_missing}')
if total_missing == 0:
    print('\n✅ Tidak ada missing values pada dataset. Tidak perlu penanganan khusus.')
else:
    print('\n⚠️ Terdapat missing values yang perlu ditangani.')

### 2.2 Pengecekan Data Duplikat

In [12]:
# Mengecek data duplikat
duplicates = df.duplicated().sum()
print(f'Jumlah data duplikat: {duplicates}')

if duplicates > 0:
    print(f'\nMenghapus {duplicates} data duplikat...')
    df = df.drop_duplicates()
    print(f'✅ Data setelah menghapus duplikat: {df.shape[0]} baris')
else:
    print('✅ Tidak ada data duplikat.')

### 2.3 Encoding Data Kategorikal

Algoritma C4.5 (Decision Tree) memerlukan input berupa data numerik. Oleh karena itu, kita perlu mengkonversi data kategorikal menjadi angka menggunakan **Label Encoding**.

Proses encoding:
- **Binary features** (yes/no, Male/Female): Dikonversi menjadi 0 dan 1
- **Ordinal features** (CAEC, CALC): Dikonversi sesuai urutan (no=0, Sometimes=1, Frequently=2, Always=3)
- **Nominal features** (MTRANS): Dikonversi menggunakan Label Encoding

In [13]:
# Membuat copy dataframe untuk preprocessing
df_encoded = df.copy()

# Menyimpan mapping encoding untuk dokumentasi
encoding_maps = {}

# 1. Encoding fitur binary (yes/no)
binary_cols = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_cols:
    df_encoded[col] = df_encoded[col].map({'yes': 1, 'no': 0})
    encoding_maps[col] = {'yes': 1, 'no': 0}

# 2. Encoding Gender
df_encoded['Gender'] = df_encoded['Gender'].map({'Male': 1, 'Female': 0})
encoding_maps['Gender'] = {'Male': 1, 'Female': 0}

# 3. Encoding fitur ordinal (CAEC dan CALC)
ordinal_map = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
for col in ['CAEC', 'CALC']:
    df_encoded[col] = df_encoded[col].map(ordinal_map)
    encoding_maps[col] = ordinal_map

# 4. Encoding MTRANS menggunakan LabelEncoder
le_mtrans = LabelEncoder()
df_encoded['MTRANS'] = le_mtrans.fit_transform(df_encoded['MTRANS'])
encoding_maps['MTRANS'] = dict(zip(le_mtrans.classes_, le_mtrans.transform(le_mtrans.classes_)))

# Menampilkan hasil encoding
print('Mapping Encoding yang digunakan:')
print('=' * 50)
for col, mapping in encoding_maps.items():
    print(f'\n📌 {col}:')
    for key, val in mapping.items():
        print(f'   {key} → {val}')

In [14]:
# Menampilkan data setelah encoding
print('Data setelah encoding (5 baris pertama):')
df_encoded.head()

### 2.4 Pemisahan Fitur (X) dan Target (y)

In [15]:
# Memisahkan fitur (X) dan target (y)
X = df_encoded.drop('NObeyesdad', axis=1)
y = df_encoded['NObeyesdad']

print(f'Dimensi fitur (X): {X.shape}')
print(f'Dimensi target (y): {y.shape}')
print(f'\nDaftar fitur yang digunakan ({X.shape[1]} fitur):')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')

### 2.5 Pembagian Data Training dan Testing

Dataset dibagi dengan rasio **80% data training** dan **20% data testing** menggunakan fungsi `train_test_split` dari scikit-learn. Parameter `stratify=y` digunakan agar proporsi setiap kelas tetap terjaga pada kedua subset.

In [16]:
# Membagi data menjadi training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Hasil Pembagian Data:')
print('=' * 40)
print(f'Data Training : {X_train.shape[0]} baris ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Data Testing  : {X_test.shape[0]} baris ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'Total Data    : {len(X)} baris')

print(f'\nDistribusi kelas pada data training:')
print(y_train.value_counts().sort_index())

print(f'\nDistribusi kelas pada data testing:')
print(y_test.value_counts().sort_index())

---
# Implementasi Algoritma C4.5 (Decision Tree)

## Penjelasan Algoritma C4.5

Algoritma **C4.5** adalah algoritma klasifikasi yang dikembangkan oleh **Ross Quinlan** sebagai penyempurnaan dari algoritma ID3. C4.5 membangun model prediksi berupa **pohon keputusan (Decision Tree)** dengan cara:

1. **Menghitung Entropy** – mengukur tingkat ketidakpastian data
2. **Menghitung Information Gain** – mengukur pengurangan entropy setelah data dibagi berdasarkan suatu atribut
3. **Memilih atribut dengan Information Gain tertinggi** sebagai node keputusan
4. **Mengulangi proses** secara rekursif hingga semua data terklasifikasi atau tidak ada atribut lagi

### Rumus Entropy:
$$Entropy(S) = -\sum_{i=1}^{n} p_i \log_2(p_i)$$

### Rumus Information Gain:
$$Gain(S, A) = Entropy(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} \cdot Entropy(S_v)$$

Pada implementasi ini, kita menggunakan `DecisionTreeClassifier` dari scikit-learn dengan parameter `criterion='entropy'` untuk menerapkan konsep C4.5.

### 3.1 Membangun Model C4.5 (Decision Tree)

In [17]:
# Membangun model Decision Tree (C4.5) dengan criterion entropy
model_c45 = DecisionTreeClassifier(
    criterion='entropy',     # Menggunakan entropy (Information Gain) → konsep C4.5
    random_state=42,         # Untuk reproducibility
    max_depth=None,          # Tidak ada batasan kedalaman pohon
    min_samples_split=2,     # Minimum sampel untuk membagi node
    min_samples_leaf=1       # Minimum sampel pada leaf node
)

# Melatih model dengan data training
model_c45.fit(X_train, y_train)

print('✅ Model C4.5 (Decision Tree) berhasil dibangun!')
print(f'\nKedalaman pohon (Tree Depth): {model_c45.get_depth()}')
print(f'Jumlah leaf nodes: {model_c45.get_n_leaves()}')
print(f'Jumlah fitur yang digunakan: {model_c45.n_features_in_}')

### 3.2 Menampilkan Parameter Model

In [18]:
# Menampilkan parameter model yang digunakan
print('Parameter Model C4.5 (Decision Tree):')
print('=' * 50)
params = model_c45.get_params()
for key, value in params.items():
    print(f'  {key:25s} : {value}')

### 3.3 Melakukan Prediksi pada Data Testing

In [19]:
# Melakukan prediksi pada data testing
y_pred = model_c45.predict(X_test)

# Menampilkan perbandingan hasil prediksi vs aktual (10 data pertama)
comparison = pd.DataFrame({
    'Aktual': y_test.values[:10],
    'Prediksi': y_pred[:10],
    'Status': ['✅ Benar' if a == p else '❌ Salah' for a, p in zip(y_test.values[:10], y_pred[:10])]
})
print('Perbandingan Hasil Prediksi vs Aktual (10 data pertama):')
print(comparison.to_string(index=False))

### 3.4 Menampilkan Aturan Decision Tree (Cuplikan)

In [20]:
# Menampilkan aturan (rules) dari Decision Tree
# Dibatasi kedalaman 3 agar mudah dibaca
print('Cuplikan Aturan Decision Tree (kedalaman maks 3):')
print('=' * 60)
tree_rules = export_text(model_c45, feature_names=list(X.columns), max_depth=3)
print(tree_rules)

---
# Evaluasi Hasil

Pada tahap ini, kita mengevaluasi performa model C4.5 menggunakan beberapa metrik:
- **Confusion Matrix** – matriks kesalahan klasifikasi
- **Accuracy** – persentase prediksi yang benar
- **Precision** – ketepatan prediksi positif
- **Recall** – sensitivitas model
- **F1-Score** – rata-rata harmonik precision dan recall

### 4.1 Confusion Matrix

In [21]:
# Menghitung Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
labels = sorted(y_test.unique())

print('Confusion Matrix:')
print(pd.DataFrame(cm, index=labels, columns=labels))

In [22]:
# Visualisasi Confusion Matrix sebagai Heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels,
            yticklabels=labels,
            linewidths=0.5,
            square=True,
            ax=ax)
ax.set_xlabel('Prediksi', fontsize=13, fontweight='bold')
ax.set_ylabel('Aktual', fontsize=13, fontweight='bold')
ax.set_title('Confusion Matrix - Algoritma C4.5 (Decision Tree)',
             fontsize=15, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### 4.2 Metrik Evaluasi (Accuracy, Precision, Recall, F1-Score)

In [ ]:
# Menghitung metrik evaluasi
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print('\n' + '=' * 50)
print('     HASIL EVALUASI MODEL C4.5 (Decision Tree)')
print('=' * 50)
print(f'  Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  Precision : {precision:.4f} ({precision*100:.2f}%)')
print(f'  Recall    : {recall:.4f} ({recall*100:.2f}%)')
print(f'  F1-Score  : {f1:.4f} ({f1*100:.2f}%)')
print('=' * 50)

### 4.3 Classification Report (Detail per Kelas)

In [24]:
# Menampilkan classification report lengkap per kelas
print('Classification Report (Detail per Kelas):')
print('=' * 70)
print(classification_report(y_test, y_pred, digits=4))

### 4.4 Interpretasi Hasil Evaluasi

Berdasarkan hasil evaluasi di atas, dapat disimpulkan bahwa:

1. **Accuracy** menunjukkan persentase keseluruhan data testing yang berhasil diprediksi dengan benar oleh model.
2. **Precision** menunjukkan dari semua data yang diprediksi sebagai kelas tertentu, berapa persen yang benar.
3. **Recall** menunjukkan dari semua data yang sebenarnya masuk kelas tertentu, berapa persen yang berhasil ditemukan oleh model.
4. **F1-Score** merupakan rata-rata harmonik dari precision dan recall, memberikan gambaran keseimbangan antara keduanya.

Model Decision Tree (C4.5) dengan parameter `criterion='entropy'` menunjukkan performa yang baik dalam mengklasifikasikan tingkat obesitas ke dalam 7 kelas.

---
# Visualisasi

Pada tahap ini, kita membuat beberapa visualisasi yang relevan dengan algoritma dan hasil analisis.

### 5.1 Visualisasi Decision Tree (Pohon Keputusan)

In [25]:
# Visualisasi Decision Tree (dibatasi kedalaman 4 agar terbaca)
fig, ax = plt.subplots(figsize=(30, 15))
plot_tree(
    model_c45,
    feature_names=list(X.columns),
    class_names=sorted(y.unique()),
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=4,  # Dibatasi kedalaman 4 agar mudah dibaca
    ax=ax
)
ax.set_title('Visualisasi Decision Tree C4.5 (Kedalaman Maks 4)',
             fontsize=20, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 5.2 Feature Importance (Tingkat Kepentingan Fitur)

In [26]:
# Menghitung dan menampilkan feature importance
feature_importance = pd.DataFrame({
    'Fitur': X.columns,
    'Importance': model_c45.feature_importances_
}).sort_values('Importance', ascending=True)

print('Feature Importance (Tingkat Kepentingan Fitur):')
print('=' * 50)
for _, row in feature_importance.sort_values('Importance', ascending=False).iterrows():
    bar = '█' * int(row['Importance'] * 50)
    print(f"  {row['Fitur']:35s} : {row['Importance']:.4f} {bar}")

print(f'\nFitur paling berpengaruh: {feature_importance.iloc[-1]["Fitur"]} '
      f'(importance: {feature_importance.iloc[-1]["Importance"]:.4f})')

In [27]:
# Visualisasi Feature Importance sebagai Horizontal Bar Chart
fig, ax = plt.subplots(figsize=(12, 8))

colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(feature_importance)))
bars = ax.barh(feature_importance['Fitur'], feature_importance['Importance'],
               color=colors, edgecolor='white', linewidth=0.5)

# Menambahkan nilai pada setiap bar
for bar, val in zip(bars, feature_importance['Importance']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Importance Score', fontsize=13, fontweight='bold')
ax.set_title('Feature Importance - Algoritma C4.5 (Decision Tree)',
             fontsize=15, fontweight='bold', pad=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### 5.3 Perbandingan Distribusi Aktual vs Prediksi

In [28]:
# Perbandingan distribusi aktual vs prediksi pada data testing
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Distribusi Aktual
actual_counts = pd.Series(y_test).value_counts().sort_index()
pred_counts = pd.Series(y_pred).value_counts().sort_index()

colors = sns.color_palette('husl', n_colors=len(actual_counts))

axes[0].barh(actual_counts.index, actual_counts.values, color=colors)
axes[0].set_title('Distribusi Kelas AKTUAL (Data Testing)',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jumlah Data')
for i, (idx, val) in enumerate(actual_counts.items()):
    axes[0].text(val + 0.5, i, str(val), va='center', fontweight='bold')

axes[1].barh(pred_counts.index, pred_counts.values, color=colors)
axes[1].set_title('Distribusi Kelas PREDIKSI (Data Testing)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jumlah Data')
for i, (idx, val) in enumerate(pred_counts.items()):
    axes[1].text(val + 0.5, i, str(val), va='center', fontweight='bold')

plt.suptitle('Perbandingan Distribusi Aktual vs Prediksi pada Data Testing',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.4 Visualisasi Metrik Evaluasi per Kelas

In [29]:
# Visualisasi metrik evaluasi per kelas
report = classification_report(y_test, y_pred, output_dict=True)

# Mengambil metrik per kelas (excluding avg rows)
class_labels = sorted(y_test.unique())
metrics_df = pd.DataFrame({
    'Kelas': class_labels,
    'Precision': [report[c]['precision'] for c in class_labels],
    'Recall': [report[c]['recall'] for c in class_labels],
    'F1-Score': [report[c]['f1-score'] for c in class_labels]
})

fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(class_labels))
width = 0.25

bars1 = ax.bar(x - width, metrics_df['Precision'], width, label='Precision',
               color='#3498db', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x, metrics_df['Recall'], width, label='Recall',
               color='#2ecc71', alpha=0.85, edgecolor='white')
bars3 = ax.bar(x + width, metrics_df['F1-Score'], width, label='F1-Score',
               color='#e74c3c', alpha=0.85, edgecolor='white')

# Menambahkan nilai pada bar
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xlabel('Kelas Obesitas', fontsize=13, fontweight='bold')
ax.set_ylabel('Skor', fontsize=13, fontweight='bold')
ax.set_title('Metrik Evaluasi per Kelas - Algoritma C4.5 (Decision Tree)',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(class_labels, rotation=30, ha='right')
ax.legend(fontsize=12)
ax.set_ylim(0, 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
# Kesimpulan

Berdasarkan implementasi dan analisis yang telah dilakukan, dapat disimpulkan:

1. **Dataset** Obesity Levels terdiri dari 2111 data dengan 16 fitur dan 7 kelas target tingkat obesitas.

2. **Preprocessing** yang dilakukan meliputi pengecekan missing values, penghapusan data duplikat, dan encoding fitur kategorikal menjadi numerik menggunakan Label Encoding dan Ordinal Encoding.

3. **Algoritma C4.5 (Decision Tree)** berhasil diimplementasikan menggunakan parameter `criterion='entropy'` (Information Gain) dari library scikit-learn. Model dilatih menggunakan 80% data training dan dievaluasi menggunakan 20% data testing.

4. **Evaluasi** model menunjukkan hasil yang baik berdasarkan metrik Accuracy, Precision, Recall, dan F1-Score. Confusion Matrix menunjukkan bahwa model mampu mengklasifikasikan sebagian besar tingkat obesitas dengan benar.

5. **Feature Importance** menunjukkan fitur-fitur yang paling berpengaruh dalam menentukan tingkat obesitas, yang dapat memberikan insight berharga untuk pemahaman lebih lanjut tentang faktor-faktor risiko obesitas.

6. Algoritma C4.5 terbukti cocok untuk dataset ini karena mampu menangani campuran data numerik dan kategorikal, serta menghasilkan model yang mudah diinterpretasikan dalam bentuk pohon keputusan.

---